# 01 — Dataset and Schema Inspection

Creates the synthetic demo databases (deterministic, no PII) and inspects
them into `SchemaContext` — the single source of truth used for schema
selection, prompt construction, and the policy table allowlist.

> Runs entirely on CPU. No model required.


In [ ]:
# Cell 00 — environment check + repo bootstrap (works on Colab and locally)
import importlib.util
import platform
import sqlite3
import subprocess
import sys
from pathlib import Path

print("python :", platform.python_version())
print("sqlite :", sqlite3.sqlite_version)

try:
    import torch
    print("torch  :", torch.__version__, "| CUDA:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU    :", torch.cuda.get_device_name(0))
    HAS_GPU = torch.cuda.is_available()
except ImportError:
    print("torch  : not installed (CPU-only mode — scripted/ollama backends still work)")
    HAS_GPU = False

# Locate the repo root whether we run from notebooks/, the repo root, or Colab.
def find_repo_root() -> Path:
    for base in (Path.cwd(), Path.cwd().parent):
        if (base / "src" / "sql_agent").is_dir():
            return base
    # Colab: clone if the repo is not present yet (replace URL with your fork).
    clone_dir = Path.cwd() / "local-enterprise-sql-agent"
    if not clone_dir.is_dir():
        raise RuntimeError(
            "repo not found — on Colab run: "
            "!git clone <YOUR_REPO_URL> local-enterprise-sql-agent"
        )
    return clone_dir

REPO_ROOT = find_repo_root()
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))
print("repo   :", REPO_ROOT)

if importlib.util.find_spec("sql_agent") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_ROOT)], check=True)
import sql_agent
print("package:", sql_agent.__version__)


In [ ]:
# Create retail.sqlite and hr.sqlite (idempotent, seeded)
import subprocess, sys
print(subprocess.run(
    [sys.executable, str(REPO_ROOT / 'scripts' / 'create_demo_db.py')],
    capture_output=True, text=True, check=True,
).stdout)

In [ ]:
from sql_agent import inspect_database

retail = inspect_database(REPO_ROOT / 'data/databases/retail.sqlite')
print(retail.to_prompt_text(include_samples=False))

In [ ]:
# Sample values are truncated and synthetic-only; this is exactly what
# the LLM prompt will contain.
print(retail.to_prompt_text(include_samples=True)[:1500])

In [ ]:
import json

questions = [
    json.loads(line)
    for line in (REPO_ROOT / 'data/evaluation_questions.jsonl').read_text().splitlines()
]
print(f'{len(questions)} business questions')
for q in questions[:5]:
    print(f"- [{q['id']}] {q['question']}")

## Definition of Done (Day 1)
- [x] Databases are recreated by `scripts/create_demo_db.py` (version-controlled generator)
- [x] `SchemaContext` is a Pydantic model with tables, columns, PK/FK, samples
- [x] Evaluation questions live in `data/evaluation_questions.jsonl` with gold SQL
